## Overview

Understanding the transformer at the code level bridges the gap between the mathematical notation
in papers and the intuition you need to debug, fine-tune, and extend these models.
In this tutorial you will build every component from scratch in pure PyTorch — no HuggingFace,
no pre-trained weights — finishing with a transformer classifier trained on synthetic sequences.

## Setup


In [ ]:
#| eval: false
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Scaled Dot-Product Attention

The fundamental operation: each query vector attends over all key-value pairs.

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$


In [ ]:
#| eval: false
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q: (batch, heads, seq_q, d_k)
        K: (batch, heads, seq_k, d_k)
        V: (batch, heads, seq_k, d_v)
        mask: (batch, 1, seq_q, seq_k) or None
    Returns:
        output: (batch, heads, seq_q, d_v)
        weights: (batch, heads, seq_q, seq_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output  = torch.matmul(weights, V)
    return output, weights


# Quick sanity check
B, H, T, d_k = 2, 4, 10, 16
Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output: {out.shape}    Weights: {w.shape}")


## Multi-Head Attention

Multiple attention heads allow the model to jointly attend to information from different
representation subspaces.


In [ ]:
#| eval: false
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim  = embed_dim
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, T, E) → (B, H, T, head_dim)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, H, T, head_dim) → (B, T, E)"""
        B, H, T, _ = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, T, self.embed_dim)

    def forward(self, query, key, value, mask=None):
        Q = self._split_heads(self.W_q(query))
        K = self._split_heads(self.W_k(key))
        V = self._split_heads(self.W_v(value))

        attn_out, self.attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_out = self._merge_heads(attn_out)
        return self.W_o(attn_out)


mha = MultiHeadAttention(embed_dim=64, num_heads=4)
x = torch.randn(2, 10, 64)
print(f"MHA output: {mha(x, x, x).shape}")     # (2, 10, 64)


## Positional Encoding

Transformers have no inherent sense of order. Sinusoidal positional encodings inject position
information as fixed, non-learnable vectors.

$$
PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)
\quad
PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)
$$


In [ ]:
#| eval: false
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Pre-compute sinusoidal table
        pe   = torch.zeros(max_len, embed_dim)
        pos  = torch.arange(0, max_len).unsqueeze(1).float()
        denom = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(pos * denom)
        pe[:, 1::2] = torch.cos(pos * denom)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, E)"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Visualise PE
pe_module = PositionalEncoding(embed_dim=64)
pe_vals = pe_module.pe[0].detach().numpy()   # (max_len, 64)

plt.figure(figsize=(12, 4))
plt.imshow(pe_vals[:50, :], aspect="auto", cmap="RdBu")
plt.colorbar()
plt.title("Sinusoidal Positional Encoding (first 50 positions, 64 dims)")
plt.xlabel("Dimension"); plt.ylabel("Position")
plt.tight_layout(); plt.show()


## Transformer Encoder Block

Each encoder block = Multi-Head Self-Attention + Feed-Forward Network, with
residual connections and layer normalisation.


In [ ]:
#| eval: false
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int,
                 ff_dim: int = None, dropout: float = 0.1):
        super().__init__()
        ff_dim = ff_dim or 4 * embed_dim

        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        # Self-attention with residual
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, mask)))
        # Feed-forward with residual
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


block = TransformerEncoderBlock(embed_dim=64, num_heads=4)
x = torch.randn(2, 10, 64)
print(f"Encoder block output: {block(x).shape}")   # (2, 10, 64)


## Transformer Classifier

Stack encoder blocks on top of an embedding layer and add a classification head.


In [ ]:
#| eval: false
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_heads: int,
                 num_layers: int, num_classes: int, max_len: int = 128,
                 dropout: float = 0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos   = PositionalEncoding(embed_dim, max_len, dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T) token ids → logits: (B, num_classes)"""
        mask = (x != 0).unsqueeze(1).unsqueeze(2)   # padding mask
        out  = self.pos(self.embed(x))              # (B, T, E)
        for layer in self.layers:
            out = layer(out, mask)
        out = self.norm(out)
        cls_repr = out[:, 0, :]                     # use [CLS] position
        return self.head(cls_repr)


model = TransformerClassifier(
    vocab_size=1000, embed_dim=64, num_heads=4,
    num_layers=2, num_classes=3
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")


## Training on Synthetic Sequences

We generate a toy binary-classification dataset: sequences starting with token 1 are class 0,
sequences starting with token 2 are class 1.


In [ ]:
#| eval: false
from torch.utils.data import TensorDataset, DataLoader

def make_synthetic_data(n: int = 2000, seq_len: int = 20, vocab_size: int = 50):
    xs, ys = [], []
    for _ in range(n):
        label = torch.randint(0, 2, (1,)).item()
        seq   = torch.randint(3, vocab_size, (seq_len,))
        seq[0] = label + 1   # class signal at position 0
        xs.append(seq); ys.append(label)
    return TensorDataset(torch.stack(xs), torch.tensor(ys))

dataset  = make_synthetic_data(2000)
train_ds, val_ds = torch.utils.data.random_split(dataset, [1600, 400])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

train_accs, val_accs = [], []

for epoch in range(1, 11):
    model.train()
    correct = total = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item(); total += len(y)
    train_accs.append(correct / total)

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item(); total += len(y)
    val_accs.append(correct / total)
    print(f"Epoch {epoch:>2}  train_acc={train_accs[-1]:.3f}  val_acc={val_accs[-1]:.3f}")


In [ ]:
#| eval: false
plt.plot(train_accs, label="Train"); plt.plot(val_accs, label="Val")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Transformer Classifier Training")
plt.legend(); plt.tight_layout(); plt.show()


## Exercises

1. **Causal Mask**: Modify `scaled_dot_product_attention` to accept a causal (upper-triangular) mask. Verify that token $i$ cannot attend to position $j > i$.

2. **Ablation: Number of Heads**: Train the classifier with `num_heads` in {1, 2, 4, 8}. How does accuracy and training time change?

3. **Layer Depth**: Compare 1, 2, and 4 encoder layers. Plot the validation accuracy curves for all three configurations on the same axes.

4. **Relative Positional Encoding**: The sinusoidal PE uses absolute positions. Research RoPE (Rotary Position Embedding) and describe conceptually how it differs.

5. **Attention Visualisation**: After training, extract `attn_weights` from the first encoder block for a sample input. Plot the (seq_len × seq_len) attention matrix as a heatmap. Which positions does token 0 (the class signal) attend to most?
